In [11]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.abstract_event_listener import AbstractEventListener
from selenium.webdriver.support.events import EventFiringWebDriver, AbstractEventListener
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import UnexpectedAlertPresentException
from selenium import webdriver
from webdriver_auto_update.chrome_app_utils import ChromeAppUtils
from webdriver_auto_update.webdriver_manager import WebDriverManager
import traceback

#* setup
def setup_chrome():
    options = Options()
    options.add_experimental_option("debuggerAddress", "localhost:8989")
    driver = webdriver.Chrome(service=Service(r'C:\bin\chromedriver.exe'), options=options)
    return driver

def get_tabs():
    global merged_dict
    try:
        # if parent.winfo_exists():
        if True:
            print("รายงานจำนวนtabs")

            # * เก็บชื่อ title และ value ของ tab ที่เปิดอยู่
            title_list = []
            # title_list_Idx = [] #!เหมือนจะไม่ได้ใช้
            value_list = []
            # title_dict = {} #!เหมือนจะไม่ได้ใช้
            for idx, handle in enumerate(driver.window_handles):
                driver.switch_to.window(handle)
                # title_list_Idx.append(
                #     driver.title + "["+str(idx)+"]") #!เหมือนจะไม่ได้ใช้
                title_list.append(driver.title)

                value_list.append(driver.current_window_handle)
                # title_dict.update(
                #     {driver.title: driver.current_window_handle}) #!เหมือนจะไม่ได้ใช้

            # * เอาtitle มาทำให้ unique เพราะ title จะสามารถที่จะซ้ำกันได้
            unique_titles = []
            counter = {}
            for item in title_list:
                if item in counter:
                    counter[item] += 1
                    print("counter[item] คือไร: ", counter[item])
                    unique_titles.append(
                        f"{item}{counter[item]-1}")
                else:
                    counter[item] = 1
                    unique_titles.append(item)

            # * เอาList มารวมกัน
            merged_dict = dict(zip(unique_titles, value_list))
            print("มี tabs ไรบ้าง", merged_dict)
            
    except Exception as e:
        traceback_str = traceback.format_exc()
        print(f"An error occirred: {e}")
        print(traceback_str)
        
def fill_items(array_items=[]):
    sku_input_xpath = '/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[1]/from/div/div/div[1]/div[1]/span/input'
    
    for item in array_items:
        driver.find_element(By.XPATH, sku_input_xpath).clear()
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(item)
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(Keys.ENTER)


driver = setup_chrome()
get_tabs()
driver.switch_to.window(merged_dict['SMCO :: เปิดการขาย'])
#* เอา function ที่ต้องการเทสมาใส่ข้างล่างนี่
item = ["CO6-010714", "CO6-010334"]
fill_items(item)


รายงานจำนวนtabs
counter[item] คือไร:  2
มี tabs ไรบ้าง {'DevTools': 'CF9036803B7DCF06F2A42AE4340576DB', 'DevTools1': 'DE717BB1E67FB51A8252B7C12C624B57', 'Seller Centre': 'B07483631DF8AC44761BDCE8FCBC8158', 'SMCO :: ลูกค้า': '848A7430A65F0DAC288FFAAA167FABCE', 'SMCO :: เปิดการขาย': '3208D062631D87DF9A099EC599C892FE'}


### PDF READER

In [ ]:
from pypdf import PdfReader
import pandas as pd
import re
from openpyxl import load_workbook

extracted_txt:str =""
target_dir = r"C:\Users\ONLINE_MIS\Downloads\TRB018324080900002-Tranfer.pdf"
reader = PdfReader(target_dir)
#* โหลดไฟล์ Excel ที่มีอยู่แล้ว
output_excel = r"C:\Users\ONLINE_MIS\Downloads\Accel_mode.xlsx"

#* สกัดเอา ข้อความออกมาจากไฟล์
for page in reader.pages:
    extracted_txt += page.extract_text()
    
pattern = r'^.*?(?=No\. Product Code Barcode Product Name Transfer No\. Order Ship Status)'
extracted_txt = re.sub(pattern, '', extracted_txt, flags=re.DOTALL)
extracted_txt = extracted_txt.lstrip()

# print(extracted_txt)


#* สกัดเอาค่าที่จำเป็นออกจากข้อความทั้งหมด
#* Regular expression สำหรับการจับ SKU
sku_pattern = r'([A-Z0-9]{3}-[0-9]{6})'

# *Regular expression สำหรับการจับ serial numbers
serial_pattern = r'Shipped\s+([\w,\s]+)(?=Serial\s*:)'

#* สกัด SKU
sku_matches = re.findall(sku_pattern, extracted_txt)

#* สกัด serial numbers
serial_matches = re.findall(serial_pattern, extracted_txt, re.DOTALL)

#* จัดการ serial numbers ให้เป็น list ของแต่ละ SKU
# serial_numbers_grouped = [serial.strip().replace('\n', '').replace(' ', '').split(',') for serial in serial_matches]
serial_numbers_grouped = [re.findall(r'\b[\w]+\b', serial) for serial in serial_matches]

# ตรวจสอบข้อมูลที่ถูกสกัด
print("SKU Matches:")
print(len(sku_matches),sku_matches)
print("Serial Numbers Grouped:")
print(len(serial_numbers_grouped), serial_numbers_grouped)

#* สร้าง DataFrame ที่แต่ละคอลัมน์เป็น SKU และแต่ละ row เป็น serial number
data = {sku: serials for sku, serials in zip(sku_matches, serial_numbers_grouped)}

# ตรวจสอบ DataFrame ก่อนเขียนลงไฟล์
print("DataFrame:")


#* เอาเข้าตาราง
try:
    # โหลด workbook และ sheet ล่าสุด
    book = load_workbook(output_excel)
    sheet = book.active

    # หาคอลัมน์ล่าสุดที่มีข้อมูล
    last_column = sheet.max_column
    
    # เขียนข้อมูลลงใน Excel
    for col, (sku, serials) in enumerate(data.items(), start=last_column+1):
        sheet.cell(row=1, column=col, value=sku)
        for row, serial in enumerate(serials, start=2):
            sheet.cell(row=row, column=col, value=serial)

    # บันทึกไฟล์
    book.save(output_excel)
    print(f"ข้อมูลถูกเพิ่มลงใน {output_excel} เรียบร้อยแล้ว")
except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")
    import traceback
    traceback.print_exc()

In [33]:
from tkinter import filedialog




def sn_extractor(output_excel, target_dir):
    extracted_txt:str = ""
    # target_dir = r"C:\Users\ONLINE_MIS\Downloads\TRB018324080900002-Tranfer.pdf" //example
    # target_dir = target_dir
    reader = PdfReader(target_dir)
    #* โหลดไฟล์ Excel ที่มีอยู่แล้ว
    # output_excel = r"C:\Users\ONLINE_MIS\Downloads\Accel_mode.xlsx" //example
    # output_excel = output_excel

    #* สกัดเอา ข้อความออกมาจากไฟล์
    for page in reader.pages:
        extracted_txt += page.extract_text()
        
    pattern = r'^.*?(?=No\. Product Code Barcode Product Name Transfer No\. Order Ship Status)'
    extracted_txt = re.sub(pattern, '', extracted_txt, flags=re.DOTALL)
    extracted_txt = extracted_txt.lstrip()

    # print(extracted_txt)


    #* สกัดเอาค่าที่จำเป็นออกจากข้อความทั้งหมด
    #* Regular expression สำหรับการจับ SKU
    sku_pattern = r'([A-Z0-9]{3}-[0-9]{6})'

    # *Regular expression สำหรับการจับ serial numbers
    serial_pattern = r'Shipped\s+([\w,\s]+)(?=Serial\s*:)'

    #* สกัด SKU
    sku_matches = re.findall(sku_pattern, extracted_txt)

    #* สกัด serial numbers
    serial_matches = re.findall(serial_pattern, extracted_txt, re.DOTALL)

    #* จัดการ serial numbers ให้เป็น list ของแต่ละ SKU
    # serial_numbers_grouped = [serial.strip().replace('\n', '').replace(' ', '').split(',') for serial in serial_matches]
    serial_numbers_grouped = [re.findall(r'\b[\w]+\b', serial) for serial in serial_matches]

    # ตรวจสอบข้อมูลที่ถูกสกัด
    print("SKU Matches:")
    print(len(sku_matches),sku_matches)
    print("Serial Numbers Grouped:")
    print(len(serial_numbers_grouped), serial_numbers_grouped)

    #* สร้าง DataFrame ที่แต่ละคอลัมน์เป็น SKU และแต่ละ row เป็น serial number
    data = {sku: serials for sku, serials in zip(sku_matches, serial_numbers_grouped)}

    # ตรวจสอบ DataFrame ก่อนเขียนลงไฟล์
    print("DataFrame:")


    #* เอาเข้าตาราง
    try:
        # โหลด workbook และ sheet ล่าสุด
        book = load_workbook(output_excel)
        sheet = book.active

        # หาคอลัมน์ล่าสุดที่มีข้อมูล
        last_column = sheet.max_column
        
        # เขียนข้อมูลลงใน Excel
        for col, (sku, serials) in enumerate(data.items(), start=last_column+1):
            sheet.cell(row=1, column=col, value=sku)
            for row, serial in enumerate(serials, start=2):
                sheet.cell(row=row, column=col, value=serial)

        # บันทึกไฟล์
        book.save(output_excel)
        print(f"ข้อมูลถูกเพิ่มลงใน {output_excel} เรียบร้อยแล้ว")
    except Exception as e:
        print(f"เกิดข้อผิดพลาด: {e}")
        import traceback
        traceback.print_exc()

def extract_sn_btn(accel_file_dir):
    if not accel_file_dir:
        print("select accel file first!!")
        return
    
    target_dirs:tuple = filedialog.askopenfilenames()
    if len(target_dirs) != 0:
        for target_dir in target_dirs:
            sn_extractor(accel_file_dir, target_dir)
    else:
        print("You have not selected any transfer file, Extraction ends!!")